# 3D-SynTree: Resilient Offline Kaggle Runner

This notebook runs in **100% Offline Mode (Internet: OFF)** on Kaggle GPU accelerators (such as NVIDIA RTX 6000 / T4 / P100).

### Capabilities:
1. **Fuzzy Path Discovery:** Automatically locates your codebase and dataset regardless of folder naming, nesting, or mount points.
2. **Offline Dependency Installation:** Installs embedded `torch-geometric` and `rdkit` wheels directly from the codebase with zero internet calls.
3. **Dynamic Configuration:** Injects verified runtime paths into `runtime_config.json` to prevent path mismatch errors.
4. **Production Execution:** Executes resilient behavioral cloning training (`main.py --mode train`).

In [ ]:
# CELL 1: Intelligent Environment Auto-Discovery & Path Resolution
import os
import sys
import re
import glob
import json
from pathlib import Path

print("=" * 70)
print("STARTING INTELLIGENT KAGGLE AUTO-DISCOVERY")
print("=" * 70)

INPUT_ROOT = Path("/kaggle/input")

# ------------------------------------------------------------------------------
# 1. LOCATE CODEBASE (looks for main.py + syntree package)
# ------------------------------------------------------------------------------
codebase_candidates = []
for root, dirs, files in os.walk(INPUT_ROOT):
    if "main.py" in files and "syntree" in dirs:
        codebase_candidates.append(Path(root))

if not codebase_candidates:
    # Fallback search by regex
    for p in INPUT_ROOT.glob("**/*"):
        if p.is_dir() and re.search(r"(codebase|3d-syntree|syntree)", p.name, re.I):
            if (p / "main.py").exists():
                codebase_candidates.append(p)

if not codebase_candidates:
    raise FileNotFoundError(
        f"Could not find 3D-SynTree codebase in {INPUT_ROOT}! "
        "Please make sure you have attached your code repository dataset under + Add Input."
    )

CODEBASE_DIR = codebase_candidates[0].resolve()
print(f"[Found] Codebase Root:      {CODEBASE_DIR}")

# ------------------------------------------------------------------------------
# 2. LOCATE SYNTHON CATALOG (.parquet)
# ------------------------------------------------------------------------------
catalog_candidates = list(INPUT_ROOT.glob("**/*enamine*.parquet"))
if not catalog_candidates:
    # Broader search for any parquet catalog
    catalog_candidates = [p for p in INPUT_ROOT.glob("**/*.parquet") if "train" not in p.name and "val" not in p.name]

if not catalog_candidates:
    raise FileNotFoundError(
        f"Could not find enamine_3d_subset.parquet in {INPUT_ROOT}! "
        "Please attach your preprocessed dataset."
    )

CATALOG_PATH = catalog_candidates[0].resolve()
print(f"[Found] Synthon Catalog:    {CATALOG_PATH}")

# ------------------------------------------------------------------------------
# 3. LOCATE TRAIN SHARDS DIRECTORY (looks for train/manifest.json)
# ------------------------------------------------------------------------------
manifest_candidates = list(INPUT_ROOT.glob("**/train/manifest.json"))
if not manifest_candidates:
    # Try searching for any manifest.json inside a 'train' folder
    manifest_candidates = [p for p in INPUT_ROOT.glob("**/manifest.json") if "train" in str(p.parent).lower()]

if not manifest_candidates:
    raise FileNotFoundError(
        f"Could not find train/manifest.json in {INPUT_ROOT}! "
        "Make sure your dataset contains the preprocessed shards."
    )

# DATA_DIR is the parent directory containing the train/ folder
DATA_DIR = manifest_candidates[0].parent.parent.resolve()
print(f"[Found] Sharded Data Dir:   {DATA_DIR}")

# ------------------------------------------------------------------------------
# 4. LOCATE TARGET POCKETS (for RL / Docking)
# ------------------------------------------------------------------------------
targets_candidates = list(INPUT_ROOT.glob("**/targets/*_pocket.pdb"))
TARGETS_DIR = targets_candidates[0].parent.resolve() if targets_candidates else (DATA_DIR / "targets")
print(f"[Found] Target Pockets Dir: {TARGETS_DIR}")

# ------------------------------------------------------------------------------
# 5. LOCATE AND INSTALL OFFLINE WHEELS
# ------------------------------------------------------------------------------
wheels = list(CODEBASE_DIR.glob("**/wheels/*.whl"))
if not wheels:
    wheels = list(INPUT_ROOT.glob("**/*.whl"))

print(f"\n[Wheels] Found {len(wheels)} embedded wheel(s):")
for w in wheels:
    print(f"  - {w.name}")

if wheels:
    wheel_str = " ".join([f'"{w}"' for w in wheels])
    !pip install --quiet --no-index --no-deps {wheel_str}

# ------------------------------------------------------------------------------
# 6. CONFIGURE PYTHON PATH & VERIFY IMPORTS
# ------------------------------------------------------------------------------
if str(CODEBASE_DIR) not in sys.path:
    sys.path.insert(0, str(CODEBASE_DIR))

import torch
import rdkit
import torch_geometric
import syntree

print("\n" + "=" * 70)
print("OFFLINE ENVIRONMENT VERIFICATION COMPLETE:")
print(f"  PyTorch:           {torch.__version__}")
print(f"  PyTorch Geometric: {torch_geometric.__version__}")
print(f"  RDKit:             {rdkit.__version__}")
print(f"  3D-SynTree:        {syntree.__version__}")
print(f"  CUDA Device:       {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print("=" * 70)

In [ ]:
# CELL 2: Dynamic Runtime Configuration Generator
# Automatically bridges discovered paths into main.py without manual editing
import json
from pathlib import Path

WORKING_DIR = Path("/kaggle/working")
RUNTIME_CONFIG_PATH = WORKING_DIR / "runtime_config.json"

runtime_payload = {
    "config_override": {
        "data": {
            "backend": "kaggle_offline",
            "data_dir": str(DATA_DIR),
            "synthon_catalog_path": str(CATALOG_PATH),
            "preload_to_ram": True
        },
        "reinforcement_learning": {
            "pocket_dir": str(TARGETS_DIR)
        }
    }
}

with open(RUNTIME_CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(runtime_payload, f, indent=2)

print(f"[Config] Generated dynamic runtime configuration at {RUNTIME_CONFIG_PATH}:")
print(json.dumps(runtime_payload, indent=2))

In [ ]:
# CELL 3: Launch Resilient Offline Behavioral Cloning Training
# Executes Stage 1 training using the Kaggle 96GB/GPU configuration
import os

# Ensure all checkpoint outputs write to Kaggle's writable disk
%cd /kaggle/working

OUTPUT_EXPERIMENTS_DIR = "/kaggle/working/experiments"
CONFIG_FILE = CODEBASE_DIR / "configs" / "train_kaggle_96gb.json"

print("=" * 70)
print("LAUNCHING 3D-SYNTREE OFFLINE TRAINING ENGINE")
print("=" * 70)

!python {CODEBASE_DIR}/main.py \
    --mode train \
    --config {CONFIG_FILE} \
    --runtime-config /kaggle/working/runtime_config.json \
    --output-dir {OUTPUT_EXPERIMENTS_DIR} \
    --resume-auto

In [ ]:
# CELL 4: Training Diagnostics & Checkpoint Inspection
import json
from pathlib import Path

metrics_file = Path("/kaggle/working/experiments/latest_metrics.json")
progress_file = Path("/kaggle/working/experiments/checkpoints/progress.json")

print("=" * 70)
print("SESSION EXECUTION SUMMARY")
print("=" * 70)

if progress_file.exists():
    with open(progress_file) as f:
        prog = json.load(f)
    print(f"Completed Epochs: {prog.get('completed_epochs', [])}")
    print(f"Last Sequential Epoch: {prog.get('last_sequential_epoch', -1)}")

if metrics_file.exists():
    with open(metrics_file) as f:
        print("\nLatest Validation Metrics:")
        print(json.dumps(json.load(f), indent=2))
else:
    print("No latest_metrics.json found yet.")